In [0]:
from pyspark.ml.classification import RandomForestClassificationModel
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings("ignore")
 
VOLUME_PATH   = "/Volumes/workspace/default/raw_datasets"
PIPELINE_PATH = f"{VOLUME_PATH}/churn_pipeline"
MODEL_PATH    = f"{VOLUME_PATH}/churn_rf_model"
FILE_PATH     = f"{VOLUME_PATH}/WA_Fn-UseC_-Telco-Customer-Churn.csv"
 
print("="*50)
print("  PHASE 5: RISK SCORING + PRIORITY TIERS")
print("="*50)

In [0]:
pipeline_model = PipelineModel.load(PIPELINE_PATH)
rf_model       = RandomForestClassificationModel.load(MODEL_PATH)
 
print(f"Pipeline loaded : {PIPELINE_PATH}")
print(f"RF model loaded : {MODEL_PATH}")
print(f"  Trees         : {rf_model.getNumTrees}")
 
# Load raw CSV — same cleaning as Phase 1
df_raw = spark.read.csv(
    FILE_PATH, header=True, inferSchema=True, nullValue=" "
)
df = df_raw \
    .withColumn("TotalCharges", F.col("TotalCharges").cast(DoubleType()))
 
median_val = df.approxQuantile("TotalCharges", [0.5], 0.01)[0]
df = df.fillna({"TotalCharges": median_val})
df = df.withColumn(
    "label", F.when(F.col("Churn") == "Yes", 1).otherwise(0)
)
 
print(f"\nRaw data loaded : {df.count():,} rows x {len(df.columns)} columns")
 
# categorical_cols — same list as Phase 3 (needed for Cell 8)
categorical_cols = [
    "gender", "Partner", "Dependents", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling",
    "PaymentMethod"
]
 
print(f"categorical_cols : {len(categorical_cols)} columns ✓")


In [0]:
print("\nScoring all customers...")
 
df_model = df.drop("Churn")                        # drop text label only
 
df_features = pipeline_model.transform(df_model)   # adds 'features'
df_scored   = rf_model.transform(df_features)      # adds probability
 
extract_prob = F.udf(lambda v: float(v[1]), DoubleType())
 
df_scored = df_scored.withColumn(
    "churn_probability",
    extract_prob(F.col("probability"))
)
 
print(f"Scored {df_scored.count():,} customers.")
print("\nSample scores:")
display(
    df_scored.select(
        "customerID", "tenure", "Contract", "MonthlyCharges",
        F.round("churn_probability", 3).alias("churn_probability"),
        "prediction"
    ).limit(10)
)

In [0]:
print("Churn probability distribution (top scores):")
display(
    df_scored.select(
        F.round("churn_probability", 2).alias("prob_rounded")
    ).groupBy("prob_rounded")
     .count()
     .orderBy(F.desc("prob_rounded"))
     .limit(20)
)
 
# Get percentile cutoffs from actual score distribution
quantiles = df_scored.approxQuantile(
    "churn_probability", [0.50, 0.70, 0.80, 0.90], 0.01
)
print(f"\nScore percentiles:")
print(f"  50th pct (median) : {quantiles[0]:.3f}")
print(f"  70th pct          : {quantiles[1]:.3f}")
print(f"  80th pct          : {quantiles[2]:.3f}")
print(f"  90th pct (top 10%): {quantiles[3]:.3f}")

In [0]:
high_threshold   = quantiles[3]   # 90th percentile
medium_threshold = quantiles[1]   # 70th percentile
 
print(f"\nTier thresholds applied:")
print(f"  HIGH   : score >= {high_threshold:.3f}  (top 10% of customers)")
print(f"  MEDIUM : score >= {medium_threshold:.3f}  (next 20% of customers)")
print(f"  LOW    : score <  {medium_threshold:.3f}  (remaining 70%)")
 
df_tiered = df_scored \
    .withColumn(
        "retention_priority",
        F.when(F.col("churn_probability") >= high_threshold,
               "HIGH - Immediate personal outreach")
        .when(F.col("churn_probability") >= medium_threshold,
              "MEDIUM - Schedule retention call")
        .otherwise(
              "LOW - Automated monitoring only")
    ) \
    .withColumn(
        "priority_rank",
        F.when(F.col("churn_probability") >= high_threshold, 1)
        .when(F.col("churn_probability") >= medium_threshold, 2)
        .otherwise(3)
    )
 
print("\nCustomers per retention tier:")
display(
    df_tiered.groupBy("retention_priority", "priority_rank")
             .count()
             .orderBy("priority_rank")
)
 

In [0]:
print("Business metrics by tier:")
display(
    df_tiered.groupBy("retention_priority").agg(
        F.count("*")
         .alias("customers"),
        F.round(F.avg("churn_probability") * 100, 1)
         .alias("avg_churn_score_pct"),
        F.round(F.avg("MonthlyCharges"), 2)
         .alias("avg_monthly_charges"),
        F.round(F.sum("MonthlyCharges"), 0)
         .alias("monthly_revenue_at_risk"),
        F.round(F.avg("tenure"), 1)
         .alias("avg_tenure_months")
    ).orderBy(
        F.when(F.col("retention_priority").startswith("HIGH"), 1)
         .when(F.col("retention_priority").startswith("MEDIUM"), 2)
         .otherwise(3)
    )
)

In [0]:
pdf_tiered = df_tiered.toPandas()
 
high_tier   = pdf_tiered[pdf_tiered["priority_rank"] == 1]
medium_tier = pdf_tiered[pdf_tiered["priority_rank"] == 2]
 
monthly_risk = high_tier["MonthlyCharges"].sum()
annual_risk  = monthly_risk * 12
saved_30pct  = annual_risk * 0.30
 
print("\n" + "="*55)
print("  RETENTION OPPORTUNITY — HEADLINE NUMBERS")
print("="*55)
print(f"  High-risk customers      : {len(high_tier):,}")
print(f"  Medium-risk customers    : {len(medium_tier):,}")
print(f"  Monthly revenue at risk  : ${monthly_risk:,.0f}")
print(f"  Annual revenue at risk   : ${annual_risk:,.0f}")
print(f"\n  If retention saves 30% of HIGH tier:")
print(f"  Estimated annual saving  : ${saved_30pct:,.0f}")
print("="*55)
print("\n→ Use these numbers in your GitHub README and resume bullet")

In [0]:
print("\nHigh-risk tier — Contract breakdown:")
print(high_tier["Contract"].value_counts(normalize=True)
                            .mul(100).round(1).to_string())
 
print("\nHigh-risk tier — Internet Service:")
print(high_tier["InternetService"].value_counts(normalize=True)
                                  .mul(100).round(1).to_string())
 
print("\nHigh-risk tier — Payment Method:")
print(high_tier["PaymentMethod"].value_counts(normalize=True)
                                .mul(100).round(1).to_string())

In [0]:
df_output = df_tiered.select(
    "customerID",
    "tenure",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "MonthlyCharges",
    F.round("churn_probability", 3).alias("churn_score"),
    "retention_priority",
    "priority_rank"
).orderBy("priority_rank", F.desc("churn_probability"))
 
print("\nTop 20 highest-risk customers (retention action list):")
display(df_output.limit(20))

In [0]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Customer Retention Analytics Dashboard",
             fontsize=16, fontweight="bold")
 
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
 
# Chart 1: Tier distribution (pie)
ax1 = fig.add_subplot(gs[0, 0])
tier_map    = {1: "HIGH", 2: "MEDIUM", 3: "LOW"}
tier_colors = ["#F44336", "#FF9800", "#4CAF50"]
tier_counts = pdf_tiered["priority_rank"].value_counts().sort_index()
ax1.pie(tier_counts, labels=[tier_map[i] for i in tier_counts.index],
        colors=tier_colors, autopct="%1.1f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax1.set_title("Customer Distribution\nby Retention Tier")
 
# Chart 2: Churn score distribution
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(pdf_tiered["churn_probability"], bins=40,
         color="#2196F3", alpha=0.8, edgecolor="white")
ax2.axvline(0.40, color="#FF9800", linestyle="--",
            linewidth=2, label="Medium threshold (0.40)")
ax2.axvline(0.70, color="#F44336", linestyle="--",
            linewidth=2, label="High threshold (0.70)")
ax2.set_title("Churn Score Distribution")
ax2.set_xlabel("Churn probability score")
ax2.set_ylabel("Number of customers")
ax2.legend(fontsize=8)
 
# Chart 3: Revenue at risk per tier
ax3 = fig.add_subplot(gs[0, 2])
rev    = pdf_tiered.groupby("priority_rank")["MonthlyCharges"].sum()
labels = ["HIGH", "MEDIUM", "LOW"]
vals   = [rev.get(1, 0), rev.get(2, 0), rev.get(3, 0)]
bars   = ax3.bar(labels, vals, color=tier_colors, alpha=0.85, edgecolor="white")
for bar in bars:
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 100,
             f"${bar.get_height():,.0f}",
             ha="center", fontsize=9, fontweight="bold")
ax3.set_title("Monthly Revenue at Risk\nby Tier")
ax3.set_ylabel("Total monthly charges ($)")
 
# Chart 4: Churn score vs tenure scatter
ax4 = fig.add_subplot(gs[1, 0])
sample    = pdf_tiered.sample(min(800, len(pdf_tiered)), random_state=42)
color_map = {1: "#F44336", 2: "#FF9800", 3: "#4CAF50"}
for rank, grp in sample.groupby("priority_rank"):
    ax4.scatter(grp["tenure"], grp["churn_probability"],
                c=color_map[rank], alpha=0.4, s=15,
                label=tier_map[rank])
ax4.set_xlabel("Tenure (months)")
ax4.set_ylabel("Churn probability")
ax4.set_title("Churn Score vs Tenure")
ax4.legend(fontsize=8)
 
# Chart 5: Monthly charges by tier boxplot
ax5 = fig.add_subplot(gs[1, 1])
pdf_tiered["tier_name"] = pdf_tiered["priority_rank"].map(tier_map)
sns.boxplot(data=pdf_tiered, x="tier_name", y="MonthlyCharges",
            order=["HIGH", "MEDIUM", "LOW"],
            palette={"HIGH": "#F44336", "MEDIUM": "#FF9800", "LOW": "#4CAF50"},
            ax=ax5)
ax5.set_title("Monthly Charges\nby Retention Tier")
ax5.set_xlabel("Tier")
ax5.set_ylabel("Monthly Charges ($)")
 
# Chart 6: Contract type by tier
ax6 = fig.add_subplot(gs[1, 2])
ct     = pdf_tiered.groupby(["tier_name", "Contract"]).size().unstack(fill_value=0)
ct     = ct.reindex(["HIGH", "MEDIUM", "LOW"])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind="bar", stacked=True, ax=ax6,
            color=["#2196F3", "#9C27B0", "#FF5722"],
            edgecolor="white", alpha=0.85)
ax6.set_title("Contract Mix\nby Retention Tier")
ax6.set_xlabel("Tier")
ax6.set_ylabel("% Customers")
ax6.set_xticklabels(["HIGH", "MEDIUM", "LOW"], rotation=0)
ax6.legend(title="Contract", fontsize=8, bbox_to_anchor=(1.05, 1))
 
plt.savefig(f"{VOLUME_PATH}/retention_dashboard.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Dashboard saved to volume.")

In [0]:
output_pdf = df_output.toPandas()
output_pdf.to_csv(
    f"{VOLUME_PATH}/retention_action_list.csv",
    index=False
)
print(f"\nSaved {len(output_pdf):,} scored customers to volume.")
print(f"File: {VOLUME_PATH}/retention_action_list.csv")
 

In [0]:
print("\n" + "="*60)
print("  RECOMMENDED RETENTION ACTIONS BY TIER")
print("="*60)
print("""
  HIGH (churn score ≥ 0.70):
    • Personal call from retention specialist within 48 hrs
    • Offer: upgrade month-to-month → 1-year contract
    • Incentive: 15% discount on MonthlyCharges for 6 months
    • Flag: tech support check if on Fiber optic
 
  MEDIUM (0.40 – 0.70):
    • Automated personalised email within 1 week
    • Offer: switch to auto-pay (lower churn risk)
    • Offer: bundle upgrade (streaming/security add-ons)
    • Schedule 30-day check-in call
 
  LOW (churn score < 0.40):
    • Standard monthly newsletter
    • Loyalty reward at tenure milestones (12, 24, 36 months)
    • Monitor for sudden MonthlyCharges increases
    • Re-score monthly — watch for tier movement upward
""")

In [0]:
total_customers = len(pdf_tiered)
 
print("="*62)
print("  PROJECT COMPLETE — YOUR RESUME BULLET")
print("="*62)
print(f"""
"Built an end-to-end Customer Retention Analytics pipeline
 using PySpark and MLlib on Databricks Free Edition.
 Applied chi-square and t-test statistical feature selection
 across {total_customers:,} customers. Trained and compared Logistic
 Regression, Random Forest, and Gradient Boosted Trees models
 with MLflow experiment tracking. Generated churn probability
 scores and 3-tier prioritisation output identifying
 ${annual_risk:,.0f} annual revenue at risk, enabling targeted
 retention outreach for high-risk segments."
""")
print("="*62)
